# Exploratory analysis: NPS recreation visits, 2022–2025

Goal: check whether the draft question ("When and where can you escape the national park crowds?") is supported by the data.

In [ ]:
import pandas as pd

# utf-8-sig strips the invisible byte-order mark at the start of the file.
# Without it, the first column is read as '\ufeffParkName' instead of 'ParkName'.
raw = pd.read_csv("../backend/data/raw/nps_visits_raw.csv", encoding="utf-8-sig")

# Keep only the 7 columns we use (the file has 35)
columns = ["ParkName", "UnitCode", "Region", "State", "Year", "Month", "RecreationVisits"]
df = raw[columns]

print(df.shape)   # expect (3024, 7): 63 parks x 48 months
print(df.dtypes)  # RecreationVisits will show as 'str' (text) until we clean it
df.head()

In [ ]:
# --- Cleaning (this cell gets reused in backend/main.py) ---

# Visits are text like "10,411": remove the commas, then convert to whole numbers
df["RecreationVisits"] = df["RecreationVisits"].str.replace(",", "").astype(int)

# Region values have trailing spaces ("Intermountain "), which would break filters
df["Region"] = df["Region"].str.strip()

# A real date (first day of each month) so time series sort and plot in order
df["Date"] = pd.to_datetime(df["Year"].astype(str) + "-" + df["Month"].astype(str) + "-01")

df.dtypes

In [ ]:
# --- Sanity checks (notebook only) ---
# Each assert stops the notebook with an error if the condition is False.

assert df["UnitCode"].nunique() == 63, "expected 63 parks"
assert (df.groupby("UnitCode").size() == 48).all(), "every park should have 48 months"
assert df.isna().sum().sum() == 0, "found missing values"
assert df.duplicated(subset=["UnitCode", "Year", "Month"]).sum() == 0, "found duplicate park-months"
assert (df["RecreationVisits"] >= 0).all(), "found negative visits"

print("Regions:", sorted(df["Region"].unique()))
print("Date range:", df["Date"].min().date(), "to", df["Date"].max().date())
print("Zero-visit months (real closures, kept):", (df["RecreationVisits"] == 0).sum())
print("All checks passed")

In [ ]:
# --- Check 1: what share of all visits do the top 10 parks get? ---

# Total visits per park over all 48 months, busiest first
park_totals = df.groupby("ParkName")["RecreationVisits"].sum().sort_values(ascending=False)

grand_total = park_totals.sum()
top10 = park_totals.head(10)
top10_share = top10.sum() / grand_total

print(f"All visits 2022-2025: {grand_total:,}")
print(f"Top 10 parks' share: {top10_share:.1%}  (an even split would be 10/63 = {10/63:.1%})")
print()
print("Top 10 parks, % of all visits:")
print((top10 / grand_total * 100).round(1))
print()
print("5 quietest parks, total visits:")
print(park_totals.tail(5))

In [ ]:
# --- Check 2: which months are busiest? ---

# Total visits per calendar month (all parks, all 4 years), as % of the year
month_totals = df.groupby("Month")["RecreationVisits"].sum()
print("Share of a year's visits by month (%):")
print((month_totals / month_totals.sum() * 100).round(1))
print()

# Does the pattern hold every year? One row per year, one column per month (millions of visits)
year_month = df.pivot_table(index="Year", columns="Month", values="RecreationVisits", aggfunc="sum")
print("Visits by year and month (millions):")
print((year_month / 1_000_000).round(1))
print()

# idxmax / idxmin return the column label (the month) holding each row's largest / smallest value
print("Busiest month each year:", year_month.idxmax(axis=1).tolist())
print("Quietest month each year:", year_month.idxmin(axis=1).tolist())